In [16]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import pandas as pd


device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [17]:
from PIL import Image
import numpy as np

img = Image.open("./dataset/0/train_30_00000.png")
bitmap = np.array(img)

print(bitmap.shape)
print(bitmap[:,50])

(128, 128, 3)
[[255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [255 255 255]
 [  0   0   0]
 [  0   0   0]
 [  0   0   0]
 [  0   0   0]
 [  0   0   0]
 [  0   0   0]
 [  0   0  

In [18]:
import os
import pandas as pd

data = []
base_dir = "./dataset"

valid_ext = (".png", ".jpg", ".jpeg", ".bmp")

for label in os.listdir(base_dir):
    label_path = os.path.join(base_dir, label)

    if os.path.isdir(label_path):

        for file in os.listdir(label_path):

            if not file.lower().endswith(valid_ext):
                continue

            file_path = os.path.join(label_path, file)

            data.append({
                "label": label[0],   # first char label if needed
                "filepath": file_path
            })

df = pd.DataFrame(data)

# encode labels
df["label_id"] = df["label"].astype("category").cat.codes

print(df.iloc[0], df.iloc[10001], df.iloc[20001])
print(df["label"].nunique())
print(df["label"].unique())
print(df["label"].value_counts())

label                                    0
filepath    ./dataset\0\train_30_00000.png
label_id                                 0
Name: 0, dtype: object label                                    1
filepath    ./dataset\1\train_31_00019.png
label_id                                 1
Name: 10001, dtype: object label                                    2
filepath    ./dataset\2\train_32_00008.png
label_id                                 2
Name: 20001, dtype: object
36
['0' '1' '2' '3' '4' '5' '6' '7' '8' '9' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h'
 'i' 'j' 'k' 'l' 'm' 'n' 'o' 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z']
label
r    21318
t    20000
n    19149
a    17010
o    16950
h    16657
e    15420
l    15390
d    14945
g    13654
u    12837
c    12792
i    12788
s    12698
m    12634
f    12493
q    12198
p    11678
0    10000
5    10000
1    10000
9    10000
8    10000
7    10000
6    10000
4    10000
3    10000
2    10000
b     9642
v     7805
w     7725
y     7447
j     5882
x     5551
z     

In [19]:
percent = df['label'].value_counts(normalize=True).mul(100)
print(percent)

label
r    5.014561
t    4.704532
n    4.504354
a    4.001204
o    3.987091
h    3.918169
e    3.627194
l    3.620137
d    3.515461
g    3.211784
u    3.019604
c    3.009019
i    3.008078
s    2.986907
m    2.971853
f    2.938686
q    2.869294
p    2.746976
0    2.352266
5    2.352266
1    2.352266
9    2.352266
8    2.352266
7    2.352266
6    2.352266
4    2.352266
3    2.352266
2    2.352266
b    2.268055
v    1.835944
w    1.817125
y    1.751732
j    1.383603
x    1.305743
z    1.275869
k    1.184366
Name: proportion, dtype: float64


In [20]:
def stratified_split(df, train_frac, val_frac, label_col):
    train_list = []
    val_list = []
    test_list = []

    for label, group in df.groupby(label_col):
        group = group.sample(frac=1, random_state=123).reset_index(drop=True)

        train_end = int(len(group) * train_frac)
        val_end = train_end + int(len(group) * val_frac)

        train_list.append(group[:train_end])
        val_list.append(group[train_end:val_end])
        test_list.append(group[val_end:])

    train_df = pd.concat(train_list).sample(frac=1, random_state=123).reset_index(drop=True)
    val_df = pd.concat(val_list).sample(frac=1, random_state=123).reset_index(drop=True)
    test_df = pd.concat(test_list).sample(frac=1, random_state=123).reset_index(drop=True)

    return train_df, val_df, test_df

In [21]:
train_df, val_df, test_df = stratified_split(df, .7, .1, "label")

In [22]:
def balance_undersample(df, label_col):
    min_count = df[label_col].value_counts().min()
    
    balanced_df = []

    for label, group in df.groupby(label_col):
        sampled = group.sample(n=min_count, random_state=123)
        balanced_df.append(sampled)

    return pd.concat(balanced_df).sample(frac=1, random_state=123).reset_index(drop=True)

In [23]:
train_df = balance_undersample(train_df, "label")

In [24]:
percent = train_df['label'].value_counts(normalize=True).mul(100)
print(percent)

label
d    2.777778
u    2.777778
7    2.777778
g    2.777778
k    2.777778
x    2.777778
2    2.777778
0    2.777778
5    2.777778
q    2.777778
v    2.777778
3    2.777778
f    2.777778
1    2.777778
c    2.777778
e    2.777778
m    2.777778
8    2.777778
s    2.777778
t    2.777778
4    2.777778
j    2.777778
h    2.777778
z    2.777778
w    2.777778
i    2.777778
b    2.777778
n    2.777778
6    2.777778
o    2.777778
l    2.777778
a    2.777778
r    2.777778
p    2.777778
9    2.777778
y    2.777778
Name: proportion, dtype: float64


In [25]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor()
])

In [26]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class DataSet(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["filepath"]).convert("L")
        img = self.transform(img)

        label = torch.tensor(row["label_id"], dtype=torch.long)

        return img, label

    def __len__(self):
        return len(self.df)

In [27]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 128 → 64

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 64 → 32

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 → 16
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [28]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = df["label_id"].nunique()

model = CNN(num_classes).to(device)  

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(device)

cuda


In [29]:
from torch.utils.data import DataLoader

dataset = DataSet(train_df, transform)
loader = DataLoader(
    dataset,
    batch_size=64,          
    shuffle=True,
    num_workers=0, 
    pin_memory=True,
)

In [30]:
epochs = 15

for epoch in range(epochs):
    count = 0
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        count+=1
        if count%500 == 0:
            print(count , len(loader))
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

500 1983
1000 1983


KeyboardInterrupt: 

In [ ]:
correct = 0
total = 0

model.eval()

val_dataset = DataSet(val_df, transform)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False, 
    num_workers=0,
    pin_memory=True
)

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Accuracy:", correct / total)

In [31]:
# adding better normalization and rotation to images in order to improve genralization
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [37]:
#adding cnn Layers along with dropout
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [38]:
#using lr scheduler to change learning rate in order to better the training process
num_classes = df["label_id"].nunique()

model = CNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [39]:
train_dataset = DataSet(train_df, train_transform)
val_dataset = DataSet(val_df, val_transform)
test_dataset = DataSet(test_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [40]:
from tqdm import tqdm
best_val_loss = float("inf")
epochs = 30

for epoch in range(epochs):

    model.train()
    train_loss = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")

    for imgs, labels in train_bar:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        train_bar.set_postfix(loss=loss.item())

    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]")

    with torch.no_grad():
        for imgs, labels in val_bar:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            val_bar.set_postfix(loss=loss.item())

    val_acc = correct / total
    val_loss = val_loss / len(val_loader)

    scheduler.step(val_loss)

    print(f"\nEpoch {epoch+1} Summary")
    print("Train Loss:", train_loss / len(train_loader))
    print("Val Loss:", val_loss)
    print("Val Acc:", val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1 [Val]: 100%|██████████| 665/665 [00:45<00:00, 14.71it/s, loss=1.74]



Epoch 1 Summary
Train Loss: 2.4940293807305296
Val Loss: 1.9772966578490752
Val Acc: 0.5457753935201525


Epoch 2 [Val]: 100%|██████████| 665/665 [00:36<00:00, 18.01it/s, loss=1.75]



Epoch 2 Summary
Train Loss: 1.7057824249647509
Val Loss: 1.7217343000541057
Val Acc: 0.6305028116985483


Epoch 3 [Val]: 100%|██████████| 665/665 [00:37<00:00, 17.57it/s, loss=1.18]


...


Epoch 28 Summary
Train Loss: 1.0546827638203367
Val Loss: 0.9910252794287259
Val Acc: 0.8787557939813181


Epoch 29 [Val]: 100%|██████████| 665/665 [00:33<00:00, 19.99it/s, loss=0.976]



Epoch 29 Summary
Train Loss: 1.0518149130283072
Val Loss: 1.002923776153335
Val Acc: 0.8807086892073128


Epoch 30 [Val]: 100%|██████████| 665/665 [00:31<00:00, 21.09it/s, loss=0.838]


Epoch 30 Summary
Train Loss: 1.0430142735208578
Val Loss: 0.9649096125050595
Val Acc: 0.887579115785511


In [41]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Test Accuracy:", correct / total)

Test Accuracy: 0.8909909698052865
